# Spotify Tracks Genre Classification

**Final Project Notebook - Python and Advanced Data Science**

This notebook combines the group work on the Spotify Tracks Dataset. The goal is to use Spotify audio features to predict the target variable `track_genre`.

**Contributions**

- **Heyi Sun:** project introduction, notebook integration, model comparison, final conclusion, and presentation story.
- **Xinran Zhang:** data understanding, data cleaning, leakage analysis, EDA, feature selection, and preprocessing.
- **Siyuan Liu:** baseline machine learning models: Logistic Regression, KNN, and Decision Tree.
- **Weiye Hu:** advanced models: Random Forest, Neural Network, Random Forest tuning, feature importance, and Top-3 Accuracy.



## 1. Project Introduction

Music streaming platforms such as Spotify contain millions of songs and continuously receive new uploads from artists around the world. To organize this large music library and provide personalized recommendations, songs need to be accurately categorized into music genres. Manual labeling is time-consuming and difficult to scale, making automatic music genre classification an important application of machine learning.


## Problem Statement

This project uses the Spotify Tracks Dataset to predict the music genre (`track_genre`) of a song based on its audio features, including danceability, energy, loudness, acousticness, speechiness, instrumentalness, tempo, and valence.

Since the dataset contains **114 different music genres**, the task becomes a challenging multi-class classification problem. Many genres share similar acoustic characteristics, making accurate genre prediction more difficult.

> **Project Summary**
>
> - Dataset: Spotify Tracks Dataset
> - Target Variable: `track_genre`
> - Number of Genres: 114
> - Task: Multi-class Classification


## Project Objectives

The objectives of this project are to:

- understand the Spotify Tracks Dataset through data exploration;
- clean and preprocess the data for machine learning;
- build and compare multiple machine learning models;
- improve the best-performing model through hyperparameter tuning;
- identify the most suitable model for music genre classification.


## Project Workflow

The project follows a standard end-to-end data science workflow. After understanding and cleaning the dataset, exploratory data analysis (EDA) is performed to identify important patterns in the data. The processed data is then used to train multiple machine learning models. Finally, the models are evaluated, compared, and the best-performing model is further optimized through hyperparameter tuning.

```text
Spotify Dataset
       |
       v
Data Understanding
       |
       v
Data Cleaning
       |
       v
Exploratory Data Analysis
       |
       v
Data Preprocessing
       |
       v
Machine Learning Models
       |
       v
Hyperparameter Tuning
       |
       v
Model Comparison
       |
       v
Conclusion
```


## Expected Outcome

By comparing multiple machine learning models, this project aims to determine which algorithm performs best for large-scale music genre classification using Spotify audio features. The findings may also provide insights into the effectiveness of different modeling approaches for multi-class classification tasks.


In [1]:
# Core libraries
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    HAS_SEABORN = True
except ModuleNotFoundError:
    sns = None
    HAS_SEABORN = False

# Machine learning libraries
# The notebook can still be read and can display the final group results
# even if scikit-learn is not installed. Re-training requires scikit-learn.
try:
    from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        classification_report,
    )
    HAS_SKLEARN = True
except ModuleNotFoundError:
    HAS_SKLEARN = False

warnings.filterwarnings("ignore")
if HAS_SEABORN:
    sns.set_theme(style="whitegrid")
else:
    plt.style.use("ggplot")
RANDOM_STATE = 42

# Set this to True if you want to re-train all models from the raw dataset.
# It may take time because the dataset has 114 classes and more than 100,000 rows.
RUN_MODEL_TRAINING = False

# Path handling: this notebook is designed to run from the project root.
DATA_PATH_CANDIDATES = [
    Path("dataset/dataset.csv"),
    Path("../dataset/dataset.csv"),
    Path("data/dataset.csv"),
    Path("~/Downloads/dataset.csv").expanduser(),
    Path("~/Desktop/datascience_python_version0/dataset.csv").expanduser(),
    Path("/Users/xiayi/Desktop/datascience_python_version0/dataset.csv"),
    Path("/Users/xiayi/Downloads/dataset.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), DATA_PATH_CANDIDATES[0])
print("Expected dataset path:", DATA_PATH)


Expected dataset path: /Users/xiayi/Downloads/dataset.csv


## 2. Dataset Overview

This section loads the Spotify dataset and checks its basic structure. The original dataset has 21 columns and 114,000 rows. The target variable is `track_genre`, which contains 114 genre classes with 1,000 rows per genre before cleaning.

The project focuses on Spotify audio features as model inputs. Metadata columns such as `track_id`, `artists`, `album_name`, and `track_name` are useful for understanding the data, but they are not used as direct model features because they identify songs rather than describe audio characteristics.


In [2]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    display(df.head())
    print("Dataset shape:", df.shape)
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": [df[col].dtype for col in df.columns],
        "missing_values": [df[col].isna().sum() for col in df.columns],
    }))
else:
    print("Dataset file was not found in this environment.")
    print("Place dataset.csv under dataset/dataset.csv to run the full notebook.")

# Expected dataset summary from the group notebook
expected_dataset_summary = pd.DataFrame({
    "item": ["rows", "columns", "target column", "number of genres", "rows per genre before cleaning"],
    "value": [114000, 21, "track_genre", 114, 1000],
})
display(expected_dataset_summary)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


Dataset shape: (114000, 21)


,column,dtype,missing_values
0,Unnamed: 0,int64,0
1,track_id,str,0
2,artists,str,1
3,album_name,str,1
4,track_name,str,1
5,popularity,int64,0
6,duration_ms,int64,0
7,explicit,bool,0
8,danceability,float64,0
9,energy,float64,0


,item,value
0,rows,114000
1,columns,21
2,target column,track_genre
3,number of genres,114
4,rows per genre before cleaning,1000


In [3]:
metadata_columns = ["track_id", "artists", "album_name", "track_name"]

spotify_audio_features = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
]

index_like_columns = ["Unnamed: 0"]
non_core_audio_columns = ["popularity", "duration_ms", "explicit"]
target_column = "track_genre"

column_groups = pd.DataFrame({
    "column_group": [
        "Metadata / identity columns",
        "Spotify audio features",
        "Index-like column",
        "Other non-core columns",
        "Target column",
    ],
    "columns": [
        ", ".join(metadata_columns),
        ", ".join(spotify_audio_features),
        ", ".join(index_like_columns),
        ", ".join(non_core_audio_columns),
        target_column,
    ],
    "modeling_decision": [
        "Exclude from model features to avoid memorization",
        "Use as the main model input features",
        "Remove during cleaning",
        "Exclude from the first modeling version",
        "Use as y label",
    ],
})
display(column_groups)


,column_group,columns,modeling_decision
0,Metadata / identity columns,"track_id, artists, album_name, track_name",Exclude from model features to avoid memorization
1,Spotify audio features,"danceability, energy, key, loudness, mode, spe...",Use as the main model input features
2,Index-like column,Unnamed: 0,Remove during cleaning
3,Other non-core columns,"popularity, duration_ms, explicit",Exclude from the first modeling version
4,Target column,track_genre,Use as y label


## 3. Data Cleaning

The cleaning process is intentionally transparent:

1. remove the unnecessary index-like column `Unnamed: 0`;
2. check missing values;
3. remove fully duplicated rows;
4. check numeric ranges;
5. analyze repeated `track_id` values because they create a data leakage risk.

After removing the index-like column and fully duplicated rows, the cleaned dataset contains **113,550 rows** and **20 columns**. Only three metadata values are missing: one value each in `artists`, `album_name`, and `track_name`. The selected audio features and the target column do not contain missing values.


In [4]:
if DATA_PATH.exists():
    df_clean = df.copy()
    unnamed_columns = [col for col in df_clean.columns if str(col).startswith("Unnamed")]
    df_clean = df_clean.drop(columns=unnamed_columns)

    missing_values = df_clean.isna().sum()
    display(missing_values[missing_values > 0])

    duplicate_rows_before = df_clean.duplicated().sum()
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)

    cleaning_summary = pd.DataFrame({
        "step": [
            "Rows before cleaning",
            "Index-like columns removed",
            "Fully duplicated rows removed",
            "Rows after cleaning",
            "Columns after cleaning",
        ],
        "value": [
            len(df),
            ", ".join(unnamed_columns) if unnamed_columns else "None",
            duplicate_rows_before,
            len(df_clean),
            df_clean.shape[1],
        ],
    })
else:
    cleaning_summary = pd.DataFrame({
        "step": [
            "Rows before cleaning",
            "Index-like columns removed",
            "Fully duplicated rows removed",
            "Rows after cleaning",
            "Columns after cleaning",
        ],
        "value": [114000, "Unnamed: 0", 450, 113550, 20],
    })

display(cleaning_summary)


artists       1
album_name    1
track_name    1
dtype: int64

,step,value
0,Rows before cleaning,114000
1,Index-like columns removed,Unnamed: 0
2,Fully duplicated rows removed,450
3,Rows after cleaning,113550
4,Columns after cleaning,20


### Data Leakage Analysis

A major issue in this dataset is that the same `track_id` can appear multiple times, sometimes with different genre labels. If we use a normal random train-test split, the same song can appear in both training and test data. This would make the test set less independent and could bias model evaluation.

The group analysis found:

- 113,550 rows after cleaning;
- 89,741 unique `track_id` values;
- 23,809 rows have repeated `track_id` values;
- 16,299 tracks appear with more than one genre;
- a normal random split creates 6,325 overlapping `track_id` values between training and test sets.

For this reason, the project uses `GroupShuffleSplit` with `track_id` as the grouping variable. This keeps all rows for the same song on only one side of the split.


In [5]:
if DATA_PATH.exists():
    track_id_summary = pd.DataFrame({
        "rows_after_cleaning": [len(df_clean)],
        "unique_track_id": [df_clean["track_id"].nunique()],
        "rows_with_repeated_track_id": [df_clean.duplicated("track_id").sum()],
        "track_ids_appearing_more_than_once": [(df_clean["track_id"].value_counts() > 1).sum()],
    })

    genres_per_track = df_clean.groupby("track_id")[target_column].nunique()
    multi_genre_summary = pd.DataFrame({
        "track_ids_with_more_than_one_genre": [(genres_per_track > 1).sum()],
        "maximum_genres_for_one_track": [genres_per_track.max()],
    })
else:
    track_id_summary = pd.DataFrame({
        "rows_after_cleaning": [113550],
        "unique_track_id": [89741],
        "rows_with_repeated_track_id": [23809],
        "track_ids_appearing_more_than_once": [16299],
    })
    multi_genre_summary = pd.DataFrame({
        "track_ids_with_more_than_one_genre": [16299],
        "maximum_genres_for_one_track": [9],
    })

display(track_id_summary)
display(multi_genre_summary)


,rows_after_cleaning,unique_track_id,rows_with_repeated_track_id,track_ids_appearing_more_than_once
0,113550,89741,23809,16299


,track_ids_with_more_than_one_genre,maximum_genres_for_one_track
0,16299,9


## 4. Exploratory Data Analysis

The EDA focuses on understanding the genre distribution, audio-feature distributions, feature correlations, and genre-level audio patterns.

The cleaned target distribution remains relatively balanced across the 114 genres. However, the leakage analysis shows that some genre labels share many identical tracks. For example, labels such as `singer-songwriter` and `songwriter`, or `reggae` and `reggaeton`, can share many tracks. This helps explain why exact genre classification is difficult.

The audio features also have different distributions and scales. Some features range from 0 to 1, while `tempo` is measured in BPM and `loudness` is measured in decibels. This supports standardization before distance-based or gradient-based models.


In [6]:
if DATA_PATH.exists():
    genre_counts = df_clean[target_column].value_counts()
    genre_summary = pd.DataFrame({
        "number_of_genres": [genre_counts.size],
        "minimum_rows_per_genre": [genre_counts.min()],
        "maximum_rows_per_genre": [genre_counts.max()],
        "average_rows_per_genre": [round(genre_counts.mean(), 2)],
    })
    display(genre_summary)

    top_bottom_genres = pd.concat([
        genre_counts.head(10).rename("count").reset_index().assign(group="largest"),
        genre_counts.tail(10).rename("count").reset_index().assign(group="smallest"),
    ])
    top_bottom_genres = top_bottom_genres.rename(columns={"index": "track_genre"})

    plt.figure(figsize=(9, 7))
    if HAS_SEABORN:
        sns.barplot(data=top_bottom_genres, x="count", y="track_genre", hue="group")
    else:
        colors = top_bottom_genres["group"].map({"largest": "#4C78A8", "smallest": "#F58518"})
        plt.barh(top_bottom_genres["track_genre"], top_bottom_genres["count"], color=colors)
        plt.gca().invert_yaxis()
    plt.title("Largest and Smallest Genre Groups After Cleaning")
    plt.xlabel("Number of rows")
    plt.ylabel("Genre")
    plt.tight_layout()
    plt.show()
else:
    display(pd.DataFrame({
        "number_of_genres": [114],
        "minimum_rows_per_genre_after_cleaning": [899],
        "maximum_rows_per_genre_after_cleaning": [1000],
        "interpretation": ["The target remains relatively balanced after duplicate removal."],
    }))


,number_of_genres,minimum_rows_per_genre,maximum_rows_per_genre,average_rows_per_genre
0,114,904,1000,996.05


In [7]:
if DATA_PATH.exists():
    plt.figure(figsize=(10, 8))
    corr = df_clean[spotify_audio_features].corr()
    if HAS_SEABORN:
        sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5)
    else:
        plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
        plt.colorbar(label="Correlation")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
        plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Correlation Heatmap of Spotify Audio Features")
    plt.tight_layout()
    plt.show()

    genre_feature_means = df_clean.groupby(target_column)[spotify_audio_features].mean()
    example_genres = [genre for genre in ["classical", "death-metal", "salsa", "sleep", "comedy"] if genre in genre_feature_means.index]
    if example_genres:
        plt.figure(figsize=(9, 5))
        example_matrix = genre_feature_means.loc[example_genres]
        if HAS_SEABORN:
            sns.heatmap(example_matrix, annot=True, fmt=".2f", cmap="YlGnBu")
        else:
            plt.imshow(example_matrix, cmap="YlGnBu", aspect="auto")
            plt.colorbar(label="Average Feature Value")
            plt.xticks(range(len(example_matrix.columns)), example_matrix.columns, rotation=45, ha="right")
            plt.yticks(range(len(example_matrix.index)), example_matrix.index)
        plt.title("Average Audio Feature Patterns for Selected Genres")
        plt.xlabel("Audio feature")
        plt.ylabel("Genre")
        plt.tight_layout()
        plt.show()
else:
    eda_findings = pd.DataFrame({
        "finding": [
            "The dataset contains 114 fine-grained genre labels.",
            "Audio features have different scales, so scaling is needed.",
            "Energy, loudness, and acousticness show meaningful relationships.",
            "Some genres are clearly different, but many genres overlap musically.",
        ],
        "meaning_for_modeling": [
            "Multi-class classification is difficult.",
            "Use StandardScaler for KNN, Logistic Regression, and Neural Network.",
            "The model can use these features, but some redundancy may exist.",
            "Moderate model scores are expected.",
        ],
    })
    display(eda_findings)


## 5. Data Preprocessing

The final modeling dataset uses 12 Spotify audio features:

`danceability`, `energy`, `key`, `loudness`, `mode`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, and `time_signature`.

Metadata columns are excluded because they can lead to memorization and do not answer the main research question. The target variable is `track_genre`.

To avoid data leakage, the train-test split is grouped by `track_id`. The scaler is fitted only on the training data and then applied to the test data.


In [8]:
def prepare_modeling_data(input_df):
    X = input_df[spotify_audio_features].copy()
    y = input_df[target_column].copy()
    groups = input_df["track_id"].copy()

    group_split = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=RANDOM_STATE,
    )
    train_index, test_index = next(group_split.split(X, y, groups=groups))

    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()
    y_train = y.iloc[train_index].copy()
    y_test = y.iloc[test_index].copy()
    groups_train = groups.iloc[train_index].copy()
    groups_test = groups.iloc[test_index].copy()

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=spotify_audio_features,
        index=X_train.index,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        columns=spotify_audio_features,
        index=X_test.index,
    )

    overlap = set(groups_train).intersection(set(groups_test))
    split_summary = pd.DataFrame({
        "split": ["train", "test"],
        "rows": [X_train.shape[0], X_test.shape[0]],
        "features": [X_train.shape[1], X_test.shape[1]],
        "unique_genres": [y_train.nunique(), y_test.nunique()],
        "unique_track_ids": [groups_train.nunique(), groups_test.nunique()],
    })

    leakage_check = pd.DataFrame({
        "split_method": ["GroupShuffleSplit by track_id"],
        "overlapping_track_ids": [len(overlap)],
        "leakage_check_passed": [len(overlap) == 0],
    })

    return X_train_scaled, X_test_scaled, y_train, y_test, split_summary, leakage_check


if DATA_PATH.exists() and HAS_SKLEARN:
    X_train_scaled, X_test_scaled, y_train, y_test, split_summary, leakage_check = prepare_modeling_data(df_clean)
else:
    split_summary = pd.DataFrame({
        "split": ["train", "test"],
        "rows": [90973, 22577],
        "features": [12, 12],
        "unique_genres": [114, 114],
        "unique_track_ids": [71792, 17949],
    })
    leakage_check = pd.DataFrame({
        "split_method": ["GroupShuffleSplit by track_id"],
        "overlapping_track_ids": [0],
        "leakage_check_passed": [True],
    })

display(split_summary)
display(leakage_check)


,split,rows,features,unique_genres,unique_track_ids
0,train,90973,12,114,71792
1,test,22577,12,114,17949


,split_method,overlapping_track_ids,leakage_check_passed
0,GroupShuffleSplit by track_id,0,True


## 6. Baseline Models

This section represents Person3's contribution. Three baseline machine learning models are built before the advanced models. The purpose of these models is to create a fair reference point: if a more complex model does not clearly improve over the baseline models, then the additional complexity may not be useful.

All baseline models use the same prepared training and test data from Person2. They are evaluated with the same metrics: Accuracy, weighted Precision, weighted Recall, weighted F1-score, and macro F1-score. Weighted metrics consider class support, while macro F1 treats all genres equally.

### Logistic Regression

Logistic Regression is selected as the baseline model because it is simple, fast, and easy to interpret. It gives the project a first reference score for genre classification. Since Logistic Regression is mainly a linear model, it can show whether simple linear relationships between Spotify audio features and genres are enough for prediction.

**Advantage:** It is efficient and provides a clear baseline.

**Limitation:** It may not capture complex non-linear relationships between audio features and 114 genre labels.

### K-Nearest Neighbors

KNN is selected because music classification has a natural similarity idea: songs with similar audio features may belong to similar genres. For example, if a song is close to many dance or electronic songs in terms of energy, tempo, and danceability, KNN may classify it into a related genre.

**Advantage:** It is intuitive and directly uses feature similarity.

**Limitation:** It can struggle when many genres overlap in the feature space, and it is sensitive to feature scaling. This is why standardized features are used.

### Decision Tree

Decision Tree is selected as an interpretable non-linear baseline. It classifies songs by learning feature-based rules, such as splits based on energy, acousticness, or speechiness.

**Advantage:** It is easy to explain and can capture non-linear patterns.

**Limitation:** A single tree can overfit and become unstable, especially in a 114-class classification problem. This motivates the use of Random Forest in the advanced-model section.


In [9]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }, model, y_pred


if RUN_MODEL_TRAINING and DATA_PATH.exists() and HAS_SKLEARN:
    baseline_results = []

    lr_result, logistic_regression_model, logistic_regression_pred = evaluate_model(
        "Logistic Regression",
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, n_jobs=-1),
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test,
    )
    baseline_results.append(lr_result)

    knn_result, knn_model, knn_pred = evaluate_model(
        "KNN",
        KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test,
    )
    baseline_results.append(knn_result)

    dt_result, decision_tree_model, decision_tree_pred = evaluate_model(
        "Decision Tree",
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test,
    )
    baseline_results.append(dt_result)

    baseline_results_df = pd.DataFrame(baseline_results).sort_values("F1_weighted", ascending=False)
else:
    baseline_results_df = pd.DataFrame([
        {"Model": "Decision Tree", "Accuracy": 0.188466, "Precision_weighted": 0.189620, "Recall_weighted": 0.188466, "F1_weighted": 0.184350, "F1_macro": 0.185237},
        {"Model": "KNN", "Accuracy": 0.165832, "Precision_weighted": 0.187064, "Recall_weighted": 0.165832, "F1_weighted": 0.160571, "F1_macro": 0.161620},
        {"Model": "Logistic Regression", "Accuracy": 0.150596, "Precision_weighted": 0.120027, "Recall_weighted": 0.150596, "F1_weighted": 0.122212, "F1_macro": 0.122672},
    ])

display(baseline_results_df)


,Model,Accuracy,Precision_weighted,Recall_weighted,F1_weighted,F1_macro
0,Decision Tree,0.188466,0.189620,0.188466,0.184350,0.185237
1,KNN,0.165832,0.187064,0.165832,0.160571,0.161620
2,Logistic Regression,0.150596,0.120027,0.150596,0.122212,0.122672


**Baseline interpretation:**

The baseline results show a clear ranking among the three basic models. Decision Tree performs best among Person3's models, followed by KNN and Logistic Regression.

Logistic Regression has the lowest F1-scores. This is understandable because the model assumes mostly linear relationships between the selected audio features and the target genre. Spotify genre classification is more complex than that: the same tempo or energy level can appear in many different genres, and genre boundaries are not linearly separated.

KNN performs better than Logistic Regression because it uses feature similarity directly. This fits the music domain to some extent, because similar songs may have similar genre labels. However, KNN is still limited because the dataset contains 114 genres and many of them overlap musically. In a dense feature space, the nearest neighbors of a song may come from related but different genres.

Decision Tree performs best among the baseline models because it can capture non-linear feature rules. For example, it can split the data based on combinations of energy, acousticness, speechiness, and tempo. However, a single Decision Tree can be unstable and can overfit. This result motivates the next step: using Random Forest, which combines many trees to improve stability and generalization.

**Model-by-model result summary:**

- **Logistic Regression:** This model gives the lowest baseline result. Its performance suggests that simple linear decision boundaries are not enough for the Spotify genre task.
- **KNN:** KNN improves over Logistic Regression because it uses similarity between songs. However, overlapping genres make nearest-neighbor classification difficult.
- **Decision Tree:** Decision Tree is the strongest baseline because it captures non-linear rules. Its limitation is that one tree can overfit, so it becomes the natural starting point for Random Forest.


## 7. Advanced Models

This section represents Person4's contribution. The advanced-model stage adds an ensemble model, a neural-network model, and feature-importance analysis. These models are compared with Person3's baseline models to test whether more advanced methods improve Spotify genre prediction.

### Random Forest

Random Forest is used as the ensemble model. It combines many Decision Trees and makes the final prediction through voting. This is useful because a single Decision Tree can overfit, while Random Forest usually gives more stable results.

**Why Random Forest?**

- It can model non-linear relationships between Spotify audio features and genres.
- It reduces the instability of a single Decision Tree.
- It usually performs well on structured tabular data.
- It provides feature importance, which helps explain which audio features are most useful.

### Neural Network

Neural Network is selected as the deep learning model required by the project. It uses multiple dense layers to learn non-linear interactions between the 12 scaled audio features and the 114 genre labels.

**Why Neural Network?**

- It can learn complex non-linear patterns.
- It provides a useful comparison against traditional machine learning models.
- It satisfies the deep learning requirement of the project.

**Limitation:** The input data is tabular high-level audio features, not raw audio or lyrics. Therefore, a simple neural network may not outperform tree-based models.

### Feature Importance

Random Forest is also used to calculate feature importance. This helps connect model performance back to music interpretation. If important features include tempo, acousticness, speechiness, danceability, valence, loudness, and energy, the model result is easier to explain musically.


In [10]:
if RUN_MODEL_TRAINING and DATA_PATH.exists() and HAS_SKLEARN:
    advanced_results = []

    rf_result, random_forest, random_forest_pred = evaluate_model(
        "Random Forest",
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test,
    )
    advanced_results.append(rf_result)

    mlp_result, neural_network, neural_network_pred = evaluate_model(
        "Neural Network",
        MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            early_stopping=True,
            random_state=RANDOM_STATE,
            max_iter=50,
            batch_size=256,
            verbose=False,
        ),
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test,
    )
    advanced_results.append(mlp_result)

    advanced_results_df = pd.DataFrame(advanced_results)

    feature_importance_df = pd.DataFrame({
        "Feature": spotify_audio_features,
        "Importance": random_forest.feature_importances_,
    }).sort_values("Importance", ascending=False)
else:
    advanced_results_df = pd.DataFrame([
        {"Model": "Random Forest", "Accuracy": 0.273730, "Precision_weighted": 0.276947, "Recall_weighted": 0.273730, "F1_weighted": 0.260302, "F1_macro": 0.261495},
        {"Model": "Neural Network", "Accuracy": 0.176507, "Precision_weighted": 0.148072, "Recall_weighted": 0.176507, "F1_weighted": 0.147078, "F1_macro": 0.147708},
    ])
    feature_importance_df = pd.DataFrame({
        "Feature": ["tempo", "acousticness", "speechiness", "danceability", "valence", "loudness", "energy", "liveness", "instrumentalness", "key", "mode", "time_signature"],
        "Importance": [0.109091, 0.106827, 0.105840, 0.105277, 0.104883, 0.103648, 0.099984, 0.093808, 0.077379, 0.063806, 0.017616, 0.011840],
    })

display(advanced_results_df)
display(feature_importance_df)


,Model,Accuracy,Precision_weighted,Recall_weighted,F1_weighted,F1_macro
0,Random Forest,0.273730,0.276947,0.273730,0.260302,0.261495
1,Neural Network,0.176507,0.148072,0.176507,0.147078,0.147708


,Feature,Importance
0,tempo,0.109091
1,acousticness,0.106827
2,speechiness,0.105840
3,danceability,0.105277
4,valence,0.104883
5,loudness,0.103648
6,energy,0.099984
7,liveness,0.093808
8,instrumentalness,0.077379
9,key,0.063806


In [11]:
plt.figure(figsize=(8, 5))
if HAS_SEABORN:
    sns.barplot(data=feature_importance_df, x="Importance", y="Feature", color="#4C78A8")
else:
    plt.barh(feature_importance_df["Feature"], feature_importance_df["Importance"], color="#4C78A8")
    plt.gca().invert_yaxis()
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Audio Feature")
plt.tight_layout()
plt.show()


**Advanced-model interpretation:**

The advanced-model results show that Random Forest is much stronger than both the baseline Decision Tree and the Neural Network. This is one of the most important findings of the project.

Random Forest improves over Decision Tree because it does not depend on only one tree structure. Instead, it trains many decision trees and combines their predictions. This makes the model more stable and reduces the risk that one tree learns accidental patterns in the training data. Since the input features are structured numerical audio features, Random Forest is a good match for the data format.

The Neural Network performs worse than Random Forest. This does not mean that neural networks are bad models in general. Rather, it means that this specific neural-network baseline is not the best fit for the current input data. The model only receives 12 high-level audio features, not raw audio, spectrograms, lyrics, or artist/context information. Deep learning models often need richer input representations or more careful architecture tuning to outperform tree-based models on tabular data.

The most important features are `tempo`, `acousticness`, `speechiness`, `danceability`, `valence`, `loudness`, and `energy`. These are musically meaningful because they describe rhythm, acoustic/electronic character, spoken content, dance style, emotional positivity, and intensity. The least important features are `mode` and `time_signature`, likely because many songs share similar values for these variables.

**Model-by-model result summary:**

- **Random Forest:** This is the strongest advanced model before tuning. It clearly improves over Decision Tree because it combines many trees instead of relying on one unstable tree.
- **Neural Network:** This model satisfies the deep learning requirement and provides an important comparison. Its weaker result shows that a simple dense network is not automatically better than tree-based methods for tabular audio features.
- **Feature Importance:** The Random Forest feature-importance output makes the model more interpretable. The important features are musically reasonable, which supports the validity of the model.


## 8. Hyperparameter Tuning

Hyperparameter tuning is used to check whether the Random Forest model can be improved by changing its main settings. The baseline Random Forest already performs well, but tuning is still important because parameters such as tree depth and the number of trees can affect overfitting, stability, and final prediction quality.

### Why Tune the Random Forest?

Random Forest has several important hyperparameters. For example, `max_depth` controls how deep each tree can grow, while `min_samples_split` and `min_samples_leaf` control how easily trees create new splits. If these values are not chosen carefully, the model may overfit or underfit.

### Why RandomizedSearchCV?

A full `GridSearchCV` would test every possible parameter combination. However, this dataset is large and has 114 target classes, so a full grid search would take a long time. For this reason, `RandomizedSearchCV` is used as a more practical tuning method. It tests a limited number of random combinations from the parameter search space.

The tuning metric is `f1_macro`, because macro F1 gives each genre equal weight. This is important for a multi-class task where we do not want evaluation to focus only on larger or easier genre classes.

The search space includes:

- `n_estimators`: 100, 200, 300
- `max_depth`: 10, 20, 30, None
- `min_samples_split`: 2, 5, 10
- `min_samples_leaf`: 1, 2, 4


In [12]:
if RUN_MODEL_TRAINING and DATA_PATH.exists() and HAS_SKLEARN:
    rf_param_distributions = {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    }

    rf_random_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        param_distributions=rf_param_distributions,
        n_iter=10,
        cv=3,
        scoring="f1_macro",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=2,
    )

    rf_random_search.fit(X_train_scaled, y_train)
    tuned_random_forest = rf_random_search.best_estimator_
    tuned_rf_pred = tuned_random_forest.predict(X_test_scaled)

    tuned_rf_result = {
        "Model": "Tuned Random Forest",
        "Accuracy": accuracy_score(y_test, tuned_rf_pred),
        "Precision_weighted": precision_score(y_test, tuned_rf_pred, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_test, tuned_rf_pred, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_test, tuned_rf_pred, average="weighted", zero_division=0),
        "F1_macro": f1_score(y_test, tuned_rf_pred, average="macro", zero_division=0),
    }
    tuning_summary = pd.DataFrame([{
        "best_params": rf_random_search.best_params_,
        "best_cv_f1_macro": rf_random_search.best_score_,
    }])
else:
    tuned_rf_result = {
        "Model": "Tuned Random Forest",
        "Accuracy": 0.275945,
        "Precision_weighted": 0.287965,
        "Recall_weighted": 0.275945,
        "F1_weighted": 0.260596,
        "F1_macro": 0.261818,
    }
    tuning_summary = pd.DataFrame([{
        "best_params": "{'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 20}",
        "best_cv_f1_macro": 0.175897,
    }])

display(tuning_summary)
display(pd.DataFrame([tuned_rf_result]))


,best_params,best_cv_f1_macro
0,"{'n_estimators': 100, 'min_samples_split': 5, ...",0.175897


,Model,Accuracy,Precision_weighted,Recall_weighted,F1_weighted,F1_macro
0,Tuned Random Forest,0.275945,0.287965,0.275945,0.260596,0.261818


**Tuning interpretation:**

The tuned Random Forest is the best model in the final comparison. However, the improvement over the baseline Random Forest is very small. The baseline Random Forest has an F1-macro of about **0.2615**, while the tuned Random Forest has an F1-macro of about **0.2618**.

This small improvement is still useful because it confirms that the Random Forest baseline was already close to its best performance under the current feature set. The tuning process also shows that model optimization was attempted in a systematic way, which is part of the project requirements.

At the same time, the small improvement suggests that changing Random Forest parameters alone is not enough to solve the main difficulty of this project. The main limitation is likely the feature set and the label structure: there are 114 fine-grained genres, many genres overlap musically, and the model only uses 12 high-level audio features. Further improvement may require stronger feature engineering, additional features, genre hierarchy information, lyrics, artist context, playlist context, or raw audio features.

**Tuned model result summary:**

- **Tuned Random Forest:** This is the final best model. It has the highest Top-1 Accuracy, Top-3 Accuracy, F1-weighted, and F1-macro among the evaluated models.
- **Why the improvement is small:** The tuned model only slightly improves over the baseline Random Forest, which means the model choice and feature limitations matter more than small parameter changes.
- **Why it is still useful:** The tuning step proves that the model was optimized systematically and that the final result was not chosen only from an untuned default model.


## 9. Model Evaluation and Comparison

The final comparison combines Person3's baseline models and Person4's advanced models. The main metrics are:

- **Accuracy:** the percentage of exact Top-1 predictions.
- **Weighted F1-score:** F1-score weighted by class support.
- **Macro F1-score:** average F1-score across all genres, treating each genre equally.
- **Top-3 Accuracy:** whether the true genre appears among the model's three most confident predictions.

Top-3 Accuracy is especially useful because exact Top-1 prediction is strict for a 114-class genre task. For example, if the model predicts `alternative`, `indie`, and `alt-rock`, while the true label is `alt-rock`, Top-1 Accuracy marks it wrong, but Top-3 Accuracy recognizes that the model found a musically close candidate.


In [13]:
combined_results_df = pd.concat(
    [
        baseline_results_df,
        advanced_results_df,
        pd.DataFrame([tuned_rf_result]),
    ],
    ignore_index=True,
).drop_duplicates(subset=["Model"], keep="last")

combined_results_df = combined_results_df.sort_values("F1_weighted", ascending=False).reset_index(drop=True)
display(combined_results_df)

plt.figure(figsize=(10, 5))
if HAS_SEABORN:
    sns.barplot(data=combined_results_df, x="Model", y="F1_weighted", color="#4C78A8")
else:
    plt.bar(combined_results_df["Model"], combined_results_df["F1_weighted"], color="#4C78A8")
plt.title("Final Model Comparison by Weighted F1-score")
plt.xlabel("Model")
plt.ylabel("Weighted F1-score")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


,Model,Accuracy,Precision_weighted,Recall_weighted,F1_weighted,F1_macro
0,Tuned Random Forest,0.275945,0.287965,0.275945,0.260596,0.261818
1,Random Forest,0.273730,0.276947,0.273730,0.260302,0.261495
2,Decision Tree,0.188466,0.189620,0.188466,0.184350,0.185237
3,KNN,0.165832,0.187064,0.165832,0.160571,0.161620
4,Neural Network,0.176507,0.148072,0.176507,0.147078,0.147708
5,Logistic Regression,0.150596,0.120027,0.150596,0.122212,0.122672


In [14]:
top3_results_df = pd.DataFrame([
    {"Model": "Random Forest", "Top3_Accuracy": 0.456261},
    {"Model": "Neural Network", "Top3_Accuracy": 0.337910},
    {"Model": "Tuned Random Forest", "Top3_Accuracy": 0.467024},
])

top1_top3_df = combined_results_df[["Model", "Accuracy"]].rename(columns={"Accuracy": "Top1_Accuracy"})
top1_top3_df = pd.merge(top1_top3_df, top3_results_df, on="Model", how="inner")
display(top1_top3_df)

plot_df = top1_top3_df.melt(
    id_vars="Model",
    value_vars=["Top1_Accuracy", "Top3_Accuracy"],
    var_name="Metric",
    value_name="Score",
)

plt.figure(figsize=(9, 5))
if HAS_SEABORN:
    sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric")
else:
    pivot_plot_df = top1_top3_df.set_index("Model")[["Top1_Accuracy", "Top3_Accuracy"]]
    x = np.arange(len(pivot_plot_df.index))
    width = 0.35
    plt.bar(x - width / 2, pivot_plot_df["Top1_Accuracy"], width, label="Top1_Accuracy")
    plt.bar(x + width / 2, pivot_plot_df["Top3_Accuracy"], width, label="Top3_Accuracy")
    plt.xticks(x, pivot_plot_df.index, rotation=20, ha="right")
    plt.legend()
plt.title("Top-1 vs Top-3 Accuracy")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 0.55)
if HAS_SEABORN:
    plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


,Model,Top1_Accuracy,Top3_Accuracy
0,Tuned Random Forest,0.275945,0.467024
1,Random Forest,0.273730,0.456261
2,Neural Network,0.176507,0.337910


**Final comparison interpretation:**

The tuned Random Forest has the best overall result:

- Top-1 Accuracy: **27.59%**
- Top-3 Accuracy: **46.70%**
- F1-weighted: **26.06%**
- F1-macro: **26.18%**

The result is much better than random guessing. With 114 possible classes, a random Top-1 guess would be about `1 / 114 = 0.88%`. Therefore, an accuracy of 27.59% is meaningful even though it may look moderate at first. The model is not simply guessing: it learns useful patterns from the Spotify audio features.

Random Forest clearly improves over the best basic model, Decision Tree. The baseline Random Forest has F1-weighted of about **0.2603**, compared with the Decision Tree's **0.1844**. This improvement is reasonable because Random Forest combines multiple Decision Trees and reduces the instability and overfitting risk of a single tree.

The tuned Random Forest only slightly improves over the baseline Random Forest. This means tuning helped, but the baseline Random Forest was already strong for the current feature set. The main bottleneck is probably not only model parameters, but also the limited feature representation and the difficulty of separating 114 overlapping genre labels.

The Neural Network performs worse than Random Forest. This is an important comparison because neural networks are often considered more complex, but more complex does not always mean better. In this project, the Neural Network only receives 12 tabular audio features. It does not receive raw audio waves, spectrograms, lyrics, artist names, or playlist context. With this kind of structured tabular input, Random Forest is often more effective because it can handle non-linear feature interactions with less tuning and less data representation complexity.

The Top-3 Accuracy result is also important. The tuned Random Forest reaches **46.70% Top-3 Accuracy**, meaning the correct genre is often among the model's top three predictions. This supports the idea that the model captures musical similarity, even when exact genre boundaries are difficult. In music, genres are not always completely separate; for example, `alternative`, `indie`, and `alt-rock` can be very close. Top-3 Accuracy therefore gives a more realistic view of model performance for this task.


## 10. Conclusion

This project used Spotify audio features to predict `track_genre` across 114 original genre labels. The workflow included data understanding, cleaning, leakage analysis, EDA, grouped preprocessing, baseline models, advanced models, hyperparameter tuning, feature importance, and final model comparison.

The most important methodological decision was to split the data by `track_id` using `GroupShuffleSplit`. This avoids leakage caused by the same song appearing in both training and test data.

The best model was the **Tuned Random Forest**. It achieved:

- **Top-1 Accuracy:** 27.59%
- **Top-3 Accuracy:** 46.70%
- **F1-macro:** 26.18%

These results show that Spotify audio features can predict music genre much better than random guessing. With 114 possible genres, random Top-1 guessing would only achieve around **0.88%** accuracy. The tuned Random Forest's **27.59%** Top-1 Accuracy is therefore meaningful. At the same time, exact classification across 114 fine-grained genres remains challenging. This is expected because many genres overlap musically, some songs appear under multiple genre labels, and the model only uses 12 high-level audio features.

The tuned Random Forest is the best model because it combines the strengths of multiple Decision Trees. A single Decision Tree is easy to interpret but can overfit and become unstable. Random Forest reduces this problem by training many trees and combining their predictions. This makes it more reliable for a dataset where genre boundaries are noisy and non-linear. The tuning step further improves the model slightly by selecting better tree settings, especially around tree depth and split conditions.

The small difference between the baseline Random Forest and the tuned Random Forest is also meaningful. It suggests that the original Random Forest was already a strong model for this feature set. Hyperparameter tuning helped, but it could not dramatically change the result because the main challenge is not only model choice. The larger challenge is that the input features are limited and the target labels are very fine-grained.

The Neural Network performed worse than Random Forest, even though it is a more complex model. This can be explained by the structure of the data. Neural networks often perform especially well when the input representation is rich, such as images, text, raw audio, or learned embeddings. In this project, the Neural Network only receives 12 numerical audio features. These features are useful, but they are already summarized and do not contain the full sound information of each track. For this kind of tabular feature dataset, tree-based ensemble models often perform better and require less architecture tuning.

The Neural Network result should therefore be interpreted carefully. It does not show that deep learning is unsuitable for music classification in general. Instead, it shows that a simple dense neural network using only 12 high-level Spotify audio features is not enough to outperform Random Forest. A more advanced deep learning approach might need raw audio, spectrograms, lyrics, artist metadata, or pre-trained audio embeddings.

The Top-3 Accuracy result gives an important additional interpretation. The tuned Random Forest reaches **46.70% Top-3 Accuracy**, which is much higher than its Top-1 Accuracy. This means that even when the model does not choose the exact correct genre as its first prediction, it often places the correct genre among its top three candidates. This is especially useful for music genres, because many labels are musically close rather than completely separate.

The feature importance result also supports the musical interpretation of the model. The most useful features included `tempo`, `acousticness`, `speechiness`, `danceability`, `valence`, `loudness`, and `energy`. These features are meaningful for genre prediction because they describe rhythm, acoustic style, spoken content, dance character, emotional tone, and intensity. Less important features such as `mode` and `time_signature` may contribute less because many songs share similar values in these variables.

For future work, model performance may improve by adding lyrics, artist information, playlist context, raw audio embeddings, genre hierarchy information, or carefully selected additional features such as popularity and duration. Another useful direction would be to evaluate hierarchical or top-k genre prediction, because music genres are often related rather than completely separate.

**Final statement:** Spotify audio features contain meaningful genre information, and ensemble tree-based models are effective for this structured dataset. Tuned Random Forest achieved the best overall performance because it captures non-linear relationships and reduces the weakness of a single Decision Tree. The Neural Network did not outperform it because the current input features are limited tabular summaries rather than rich audio representations. Top-3 Accuracy further shows that even when the exact top prediction is incorrect, the model often identifies musically similar genre candidates.
